In [1]:
import random
import collections
    
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as D
import numpy as np
import matplotlib.pyplot as plt
from IPython import display as disp

import rldurham as rld

rld.seed_everything(42)
if torch.cuda.is_available():
    print('It is recommended to train on the CPU for this')

Seed set to 42


In [6]:
env = rld.make('CartPole-v1')              # easy discrete
# env = rld.make("LunarLander-v2")           # discrete
# env = rld.make('Breakout-v0')              # discrete
# env = rld.make("Pong-ram-v0")              # discrete
# env = rld.make("Gravitar-ram-v0")          # hard discrete
#
# env = rld.make("Pendulum-v0")              # easy continuous
# env = rld.make("BipedalWalker-v3")         # continuous
# env = rld.make("LunarLanderContinuous-v2") # continuous
# env = rld.make("BipedalWalkerHardcore-v3") # hard continuous
rld.env_info(env, print_out=True)

actions are discrete with 2 dimensions/#actions
observations are continuous with 4 dimensions/#observations
maximum timesteps is: 500


(True, False, np.int64(2), 4)

In [8]:
class Plotter:

    # data are over different 'outer' functions and multiple episodes
    TrackedValue = collections.namedtuple("TrackedValue", "inner outer data")

    def __init__(self, plot_interval=1, line_styles=("-", "--", "-.", ":"), multicol=False):
        # parameters
        self.plot_interval = plot_interval
        self.line_styles = line_styles
        self.multicol = multicol
        # track values over multiple episodes
        self.ep_vals = {}      # a dict with current values in episode (updated using 'inner')
        self.ep_val_lists = {} # a dict with values from multiple past episdes
        self.track_auc = {}    # whether to track AUCs for different values (dict)
        self.inner_func = {}   # inner function for different values (dict)
        self.outer_func = {}   # outer function(s) for different values (dict)
        # remember info from multiple experiments
        self.trace_values = []# a list of dicts of dict of trace values
        self.plot_data = []   # a list of dicts of TrackedValues
        self.plot_labels = [] # a list of labels

    def _set_dict(self, d, k, v):
        if k not in d:
            d[k] = v
        elif d[k] != v:
            raise ValueError(f"Value '{v}' for key '{k}' conflicts with stored value '{d[k]}'")

    def track(self, inner="sum", outer="mean", auc=False, **kwargs):
        for k, v in kwargs.items():
            if inner == "sum":
                if k not in self.ep_vals:
                    self.ep_vals[k] = 0
                self.ep_vals[k] += v
            elif inner == "min":
                if k not in self.ep_vals:
                    self.ep_vals[k] = v
                self.ep_vals[k] = min(self.ep_vals[k], v)
            elif inner == "max":
                if k not in self.ep_vals:
                    self.ep_vals[k] = v
                self.ep_vals[k] = max(self.ep_vals[k], v)
            elif inner == "mean":
                if k not in self.ep_vals:
                    self.ep_vals[k] = np.array([0., 0.])
                self.ep_vals[k] += [v, 1]
            else:
                raise ValueError(f"Tracking function '{inner}' is not implemented")
            self._set_dict(self.inner_func, k, inner)
            self._set_dict(self.outer_func, k, outer)
            self._set_dict(self.track_auc, k, auc)

    def track_trace_value(self, key, **kwargs):
        if key not in self.trace_values[-1]:
            self.trace_values[-1][key] = {}
        for k, v in kwargs.items():
            self.trace_values[-1][key][k] = v

    def finish_episode(self, episode, plot=None, condense=None):
        for k, v in self.ep_vals.items():
            if k not in self.ep_val_lists:
                self.ep_val_lists[k] = []
            if self.inner_func[k] == "mean":
                v = v[0] / v[1]
            self.ep_val_lists[k].append(v)
            if self.track_auc[k]:
                if k not in self.trace_values[-1]:
                    self.trace_values[-1][k] = {}
                if 'auc' not in self.trace_values[-1][k]:
                    self.trace_values[-1][k]['auc'] = 0
                self.trace_values[-1][k]['auc'] += v
        self.ep_vals = {}
        interval_match = episode % self.plot_interval == 0
        if condense or (condense is None and interval_match):
            self.condense_episode(episode)
        if plot or (plot is None and interval_match):
            self.plot()
        return interval_match

    def condense_episode(self, episode):
        for k, v in self.ep_val_lists.items():
            new_data = [episode]
            outer_funcs = []
            for func in self.outer_func[k].split("|"):
                outer_funcs.append(func)
                if func == "mean":
                    new_data.append(np.array(v).mean())
                elif func == "min":
                    new_data.append(np.array(v).min())
                elif func == "max":
                    new_data.append(np.array(v).max())
                elif func == "std":
                    new_data.append(np.array(v).std())
                else:
                    raise ValueError(f"Outer function '{func}' not implemented")
            if k not in self.plot_data[-1]:
                self.plot_data[-1][k] = self.TrackedValue(inner=self.inner_func[k],
                                                          outer=outer_funcs,
                                                          data=[])
            self.plot_data[-1][k].data.append(new_data)
        self.ep_val_lists = {}

    def plot(self, show=True):
        fig = plt.figure()
        for ex_data, ex_trace_values, ex_label in zip(self.plot_data,
                                              self.trace_values,
                                              self.plot_labels):
            # if not self.multicol:
            #     # color=next(plt.gca()._get_lines.prop_cycler)['color']
            #     # ls_cycle = itertools.cycle(self.line_styles)
            # else:
            #     plt.gca().set_prop_cycle(None)
            for value_name, tracked_value in ex_data.items():
                # if self.multicol:
                #     # color=next(plt.gca()._get_lines.prop_cycler)['color']
                #     # ls_cycle = itertools.cycle(self.line_styles)
                d = tracked_value.data
                for idx, outer in enumerate(tracked_value.outer):
                    # ls = next(ls_cycle)
                    label = "" if ex_label is None else f"{ex_label}: "
                    label += f"{value_name}-{tracked_value.inner}-{outer}"
                    if outer == "std":
                        plt.fill_between([x[0] for x in d],
                                         [x[idx] - x[idx + 1] for x in d],
                                         [x[idx] + x[idx + 1] for x in d],
                                         alpha=0.05,
                                         # color=color
                                        )
                    else:
                        tv_label = ""
                        if value_name in ex_trace_values:
                            for k, v in ex_trace_values[value_name].items():
                                if tv_label:
                                    tv_label += ", "
                                if k == 'auc':
                                    tv_label += f"AUC={np.format_float_scientific(v, precision=1)}"
                                else:
                                    tv_label += f"{k}={v}"
                            if tv_label:
                                tv_label = f" ({tv_label})"
                        plt.plot([x[0] for x in d],
                                 [x[idx + 1] for x in d],
                                 '-',
                                 # color=color,
                                 # ls=ls,
                                 label=label+tv_label)
        plt.xlabel('Episode number')
        plt.ylabel('Episode reward')
        plt.legend()
        disp.clear_output(wait=True)
        if show:
            plt.show()
        return fig

    def next_experiment(self, label=None):
        self.ep_vals = {}
        self.ep_val_lists = {}
        self.track_auc = {}
        self.inner_func = {}
        self.outer_func = {}
        self.trace_values.append({})
        self.plot_data.append({})
        self.plot_labels.append(label)


## REINFORCE (CartPole)

In [9]:
class REINFORCE(nn.Module):
    def __init__(self, env):
        discrete_act, discrete_obs, act_dim, obs_dim = rld.env_info(env)
        # discrete, obs_dim, act_dim = env_info(env, p=False)
        assert discrete_act, "REINFORCE only works for discrete action spaces"
        super().__init__()
        self.data = []
        self.fc1 = nn.Linear(obs_dim, 128)
        self.fc2 = nn.Linear(128, act_dim)

        self.optimizer = torch.optim.Adam(self.parameters(),lr=0.00002)

    def __str__(self):
        return "REINFORCE"

    def print_params(self):
        return dict()

    def sample_action(self,state):
        x = F.relu(self.fc1(torch.from_numpy(state).float()))
        prob = F.softmax(self.fc2(x),dim=0)
        action = D.Categorical(prob).sample()
        return action.item(), torch.log(prob[action])

    def put_data(self,item):
        self.data.append(item)

    def train(self):
        R = 0
        self.optimizer.zero_grad()
        for _, _, r, _, _, log_prob in self.data[::-1]:
            R = r+0.98*R
            loss = -log_prob * R
            loss.backward()
        self.optimizer.step()
        self.data = []

# Experiments

In [12]:
def experiment(env, env_name, agent, plotter, max_episodes, seed=None):
    # initialse new plot trace
    plotter.next_experiment(f"{agent} on {env_name}")
    # training procedure
    for episode in range(1, max_episodes + 1):
        # reset environment and collect new episode
        state, _ = env.reset()
        done = False
        while not done:
            # select action
            action, log_prob = agent.sample_action(state)
            # take action in environment and get r and s'
            next_state, reward, term, trun, info = env.step(action)
            done = term or trun
            agent.put_data((state, action, reward, next_state, done, log_prob))
            state = next_state
            # track performance
            plotter.track(reward=reward, auc=True) # cumulative reward
            if isinstance(agent, SAC):
                alpha = np.format_float_scientific(agent.pi.log_alpha.exp().item(), precision=2)
                plotter.track_trace_value("reward", alpha=alpha)
                plotter.track_trace_value("reward",
                                          lr_pi=agent.lr_pi,
                                          lr_q=agent.lr_q,
                                          init_alpha=agent.init_alpha,
                                          tau=agent.tau,
                                          lr_alpha=agent.lr_alpha)
        # train agent and plot progress
        agent.train()
        plotter.finish_episode(episode)

In [10]:
plotter = Plotter(plot_interval=10)

In [13]:
env_name='CartPole-v1'
env=rld.make(env_name)
agent=REINFORCE(env)
max_episodes = 5000
experiment(env,env_name,agent,plotter,max_episodes)

NameError: name 'SAC' is not defined